In [ ]:
# assign these paths for inference
SAGITTAL_DIR = r""  # e.g. r"C:\data\mouse\sagittal"
AXIAL_DIR = r""     # e.g. r"C:\data\mouse\axial"
CORONAL_DIR = r""   # e.g. r"C:\data\mouse\coronal"
ATLAS_PATH = r""  # e.g. r"C:\data\atlas\average_template_25.nrrd"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import random, urllib.request
import time
import cv2
import warnings
from scipy.spatial import ConvexHull
import tifffile
import SimpleITK as sitk
from scipy.ndimage import binary_fill_holes
from skimage.filters import threshold_li, threshold_otsu
from skimage.morphology import closing, disk
from skimage.measure import label, regionprops
from skimage.transform import resize
from skimage.metrics import hausdorff_distance
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=Image.DecompressionBombWarning)

print("Imports OK.")

In [ ]:
AXIS_INPUT_DIRS = {
    "sagittal": Path(SAGITTAL_DIR) if SAGITTAL_DIR else None,
    "axial": Path(AXIAL_DIR) if AXIAL_DIR else None,
    "coronal": Path(CORONAL_DIR) if CORONAL_DIR else None,
}

SAMPLES_PER_DATASET = 12
SUPPORTED = {".png", ".jpg", ".jpeg", ".jfif", ".bmp", ".tif", ".tiff", ".czi"}

PROCESS_SIZE = 256

AXIS_SLICE_SEARCH = {
    "coronal":  (0.05, 0.95, 15),
    "sagittal": (0.05, 0.95, 15),
    "axial":    (0.05, 0.95, 25),
}
AXIS_DIM = {"sagittal": 2, "coronal": 0, "axial": 1}

if not ATLAS_PATH.exists():
    print("Downloading Allen Mouse Atlas (~500 MB)…")
    urllib.request.urlretrieve(
        "https://download.alleninstitute.org/informatics-archive/"
        "current-release/mouse_ccf/average_template_25.nrrd",
        ATLAS_PATH)
    print("Done.")
else:
    print("Found existing atlas.")

_sitk = sitk.ReadImage(str(ATLAS_PATH))
_ori = sitk.DICOMOrientImageFilter()
_ori.SetDesiredCoordinateOrientation("RAS")
_sitk = _ori.Execute(_sitk)
_sitk = sitk.PermuteAxes(_sitk, [2, 1, 0])
atlas_vol = sitk.GetArrayFromImage(_sitk).astype(np.float32)
print(f"Atlas shape: {atlas_vol.shape}  (coronal × axial × sagittal)")


def _resize_to(arr: np.ndarray, size: int) -> np.ndarray:
    """Resize longest edge to `size`, preserving aspect ratio."""
    h, w = arr.shape
    scale = size / max(h, w)
    new_h, new_w = max(1, int(h * scale)), max(1, int(w * scale))
    return resize(arr, (new_h, new_w), anti_aliasing=True, order=1).astype(np.float32)


def get_atlas_candidates(axis: str) -> list[np.ndarray]:
    start, end, n = AXIS_SLICE_SEARCH[axis]
    dim = AXIS_DIM[axis]
    fracs = np.linspace(start, end, n)
    slices = []
    for frac in fracs:
        idx = int(atlas_vol.shape[dim] * frac)
        sl = atlas_vol[idx] if dim == 0 else (atlas_vol[:, idx] if dim == 1 else atlas_vol[:, :, idx])
        if axis == "sagittal":
            sl = np.rot90(sl, k=1)
            sl = np.fliplr(sl)
        else:
            sl = np.rot90(sl, k=2)
        sl = _resize_to(sl.astype(np.float32), PROCESS_SIZE)
        slices.append(sl)
    return slices


ATLAS_CANDIDATES = {axis: get_atlas_candidates(axis) for axis in AXIS_DIM}
ATLAS_RAW = {axis: ATLAS_CANDIDATES[axis][len(ATLAS_CANDIDATES[axis]) // 2] for axis in AXIS_DIM}

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for i, axis in enumerate(["coronal", "sagittal", "axial"]):
    sl = ATLAS_RAW[axis]
    axs[i].imshow((sl - sl.min()) / (sl.max() - sl.min() + 1e-8), cmap="gray")
    axs[i].set_title(f"Atlas — {axis}"); axs[i].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
def normalize(arr: np.ndarray) -> np.ndarray:
    lo, hi = float(arr.min()), float(arr.max())
    return (arr - lo) / (hi - lo + 1e-8)


def load_slice(path) -> np.ndarray:
    path = Path(path); ext = path.suffix.lower()
    if ext in (".png", ".jpg", ".jpeg", ".jfif", ".bmp"):
        img = Image.open(path)
        if ext in (".jpg", ".jpeg"):
            img.draft("L", (PROCESS_SIZE * 2, PROCESS_SIZE * 2))
        arr = np.array(img.convert("L"), dtype=np.float32)
    elif ext in (".tif", ".tiff"):
        arr = tifffile.imread(str(path)).astype(np.float32)
        if arr.ndim == 3:
            arr = arr.mean(2) if arr.shape[2] <= 4 else arr[arr.shape[0] // 2]
    elif ext == ".czi":
        import czifile
        arr = czifile.imread(str(path)).squeeze().astype(np.float32)
        if arr.ndim == 3:
            arr = arr.mean(2) if arr.shape[2] <= 4 else arr[arr.shape[0] // 2]
    else:
        raise ValueError(f"Unsupported format: {ext}")

    return _resize_to(arr, PROCESS_SIZE)


def is_valid_slice(path) -> bool:
    try:
        ext = Path(path).suffix.lower()
        if ext in (".png", ".jpg", ".jpeg", ".jfif", ".bmp"):
            with Image.open(path) as img:
                w, h = img.size
                return min(h, w) >= 64
        elif ext in (".tif", ".tiff"):
            with tifffile.TiffFile(str(path)) as tif:
                h, w = tif.pages[0].shape[:2]
                return min(h, w) >= 64
        return True
    except Exception:
        return False

In [ ]:
def detect_polarity(img: np.ndarray) -> int:
    n = normalize(img); h, w = n.shape
    s = max(h//8, w//8, 10)
    border = np.concatenate([n[:s,:s].ravel(), n[:s,-s:].ravel(),
                              n[-s:,:s].ravel(), n[-s:,-s:].ravel()])
    return 1 if np.median(n[h//4:3*h//4, w//4:3*w//4]) >= np.median(border) else -1


def get_tissue_mask(img: np.ndarray, min_area_frac: float = 0.02) -> np.ndarray:
    img_n = normalize(img)
    img_w = img_n if detect_polarity(img_n) == 1 else (1.0 - img_n)
    try:
        thresh = float(threshold_li(img_w))
    except:
        thresh = float(threshold_otsu(img_w))
    mask = closing(img_w > thresh, disk(5))
    mask = binary_fill_holes(mask)
    labeled = label(mask)
    if labeled.max() == 0:
        return mask.astype(bool)
    props = regionprops(labeled)
    min_area = max(int(img_n.size * min_area_frac), 200)
    big = sorted([p for p in props if p.area > min_area], key=lambda p: -p.area)
    if not big:
        big = [max(props, key=lambda p: p.area)]
    clean = np.zeros(mask.shape, bool)
    for p in big[:3]:
        clean[labeled == p.label] = True
    return clean

In [ ]:
def _ncc(a: np.ndarray, b: np.ndarray) -> float:
    a = a - a.mean(); b = b - b.mean()
    return float((a * b).sum() / (np.sqrt((a ** 2).sum() * (b ** 2).sum()) + 1e-8))


def _ncc_polarity_aware(inp: np.ndarray, atlas: np.ndarray) -> float:
    inp_r = normalize(resize(inp.astype(np.float32), atlas.shape, anti_aliasing=True, order=1))
    atlas_n = normalize(atlas.astype(np.float32))
    if detect_polarity(inp) != detect_polarity(atlas_n):
        inp_r = 1.0 - inp_r
    return _ncc(inp_r, atlas_n)


def principal_axes_angle(mask: np.ndarray) -> float:
    coords = np.column_stack(np.where(mask))
    if len(coords) < 50:
        return 0.0
    pca = PCA(n_components=2).fit(coords)
    vec = pca.components_[0]
    return np.degrees(np.arctan2(-vec[0], vec[1]))


def best_rotation(img: np.ndarray, atlas_slice: np.ndarray):
    atlas_mask = get_tissue_mask(atlas_slice)
    target_angle = principal_axes_angle(atlas_mask)

    img_mask = get_tissue_mask(img)
    brain_angle = principal_axes_angle(img_mask)

    best_score, best_k = -np.inf, 0
    nccs = []
    for k in range(4):
        candidate = np.rot90(img, k=k)
        ncc_score = _ncc_polarity_aware(candidate, atlas_slice)
        nccs.append(ncc_score)
        cand_mask = np.rot90(img_mask, k=k)
        cand_angle = principal_axes_angle(cand_mask)
        residual = (cand_angle - target_angle + 180) % 360 - 180
        pca_score = np.cos(np.radians(residual))
        combined = 0.7 * pca_score + 0.3 * ncc_score
        if combined > best_score:
            best_score, best_k = combined, k

    sorted_nccs = sorted(nccs, reverse=True)
    margin = float(sorted_nccs[0] - sorted_nccs[1])

    rotated = np.rot90(img, k=best_k)
    return rotated, best_k, float(best_k * 90), float(nccs[best_k]), margin


def resolve_flip(img: np.ndarray, atlas_slice: np.ndarray):
    candidates = {"none": img, "lr": np.fliplr(img), "ud": np.flipud(img), "both": np.flipud(np.fliplr(img))}
    best_ncc, best_key = -np.inf, "none"
    for key, cand in candidates.items():
        score = _ncc_polarity_aware(cand, atlas_slice)
        if score > best_ncc:
            best_ncc, best_key = score, key
    return candidates[best_key], (best_key if best_key != "none" else None)


def sagittal_geometric_resolve(img):
    mask = get_tissue_mask(img).astype(np.uint8)
    intensity = normalize(img)

    def score_orientation(m, inten):
        contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not contours:
            return 0.0
        cnt = max(contours, key=cv2.contourArea)
        h, w = m.shape

        hull_idx = cv2.convexHull(cnt, returnPoints=False)
        protrusion_score = 0.0
        try:
            defects = cv2.convexityDefects(cnt, hull_idx)
            if defects is not None:
                depths = defects[:, 0, 3] / 256.0
                max_depth = float(depths.max())
                deepest_idx = defects[:, 0, 2][depths.argmax()]
                deepest_pt = cnt[deepest_idx, 0]
                dx, dy = deepest_pt[0] / w, deepest_pt[1] / h
                br_score = dx * dy
                protrusion_score = (max_depth / max(h, w)) * br_score
        except:
            pass

        return protrusion_score * 10.0

    candidates = {
        "none": (mask, intensity),
        "rot180": (np.rot90(mask, k=2).astype(np.uint8), np.rot90(intensity, k=2)),
        "lr": (np.fliplr(mask).astype(np.uint8), np.fliplr(intensity)),
        "rot180lr": (np.fliplr(np.rot90(mask, k=2)).astype(np.uint8), np.fliplr(np.rot90(intensity, k=2))),
    }

    best_key = max(candidates, key=lambda k: score_orientation(candidates[k][0], candidates[k][1]))

    if best_key == "none":
        return img, None
    elif best_key == "rot180":
        return np.ascontiguousarray(np.rot90(img, k=2)), "rot180"
    elif best_key == "lr":
        return np.ascontiguousarray(np.fliplr(img)), "lr"
    else:
        return np.ascontiguousarray(np.fliplr(np.rot90(img, k=2))), "rot180lr"


def find_best_atlas_slice(img: np.ndarray, axis: str) -> np.ndarray:
    candidates = ATLAS_CANDIDATES[axis]

    fast_scores = [(i, _ncc_polarity_aware(img, sl)) for i, sl in enumerate(candidates)]
    fast_scores.sort(key=lambda x: -x[1])
    top_indices = [i for i, _ in fast_scores[:3]]

    best_ncc, best_sl = -np.inf, ATLAS_RAW[axis]
    for i in top_indices:
        sl = candidates[i]
        ncc = max(_ncc_polarity_aware(np.rot90(img, k=k), sl) for k in range(4))
        if ncc > best_ncc:
            best_ncc, best_sl = ncc, sl
    return best_sl


def randomize_orientation(img: np.ndarray, rng: np.random.Generator | None = None):
    rng = rng or np.random.default_rng()
    k = int(rng.integers(0, 4))
    out = np.rot90(img, k=k)
    flip_lr = bool(rng.random() < 0.5)
    flip_ud = bool(rng.random() < 0.5)
    if flip_lr:
        out = np.fliplr(out)
    if flip_ud:
        out = np.flipud(out)
    return out, {"k": k, "flip_lr": flip_lr, "flip_ud": flip_ud}

In [ ]:
def compute_metrics(reoriented: np.ndarray, atlas_slice: np.ndarray,
                    ncc_margin: float) -> dict:
    pred_mask  = get_tissue_mask(reoriented)
    atlas_mask = get_tissue_mask(atlas_slice)

    pred_r  = (resize(pred_mask.astype(np.float32), atlas_mask.shape,
                      anti_aliasing=False, order=0) > 0.5).astype(np.uint8)
    atlas_r = atlas_mask.astype(np.uint8)

    inter   = int((pred_r & atlas_r).sum())
    dice    = (2.0 * inter) / (int(pred_r.sum()) + int(atlas_r.sum()) + 1e-8)

    try:
        hd      = hausdorff_distance(pred_r, atlas_r)
        hd_norm = hd / np.sqrt(atlas_r.shape[0]**2 + atlas_r.shape[1]**2)
    except Exception:
        hd_norm = 1.0

    ncc = _ncc_polarity_aware(reoriented, atlas_slice)

    return {
        "dice":           round(float(dice),       4),
        "hausdorff_norm": round(float(hd_norm),    4),
        "ncc":            round(float(ncc),        4),
        "ncc_margin":     round(float(ncc_margin), 4),
    }

In [ ]:
found_axes: dict[str, list[Path]] = {axis: [] for axis in ["coronal", "sagittal", "axial"]}

for axis in ["coronal", "sagittal", "axial"]:
    p = AXIS_INPUT_DIRS.get(axis)
    if p is None:
        print(f"Missing axis folder: {axis} -> (not set)")
        continue
    if p.exists():
        found_axes[axis].append(p)
    else:
        print(f"Missing axis folder: {axis} -> {p}")

print("Found axes:")
for axis, folders in found_axes.items():
    if not folders:
        print(f"  {axis:10s}  (no folders found)")
        continue
    for folder in folders:
        n = len([f for f in folder.iterdir() if f.suffix.lower() in SUPPORTED])
        print(f"  {axis:10s}  {folder}  ({n} files)")

In [ ]:
samples: dict[str, list] = {}
samples_store: dict[str, list[Path]] = {}
rng = np.random.default_rng(42)

for axis, folders in found_axes.items():
    axis_samples = []
    axis_store: list[Path] = []

    for folder in folders:
        files = [f for f in folder.iterdir() if f.suffix.lower() in SUPPORTED and is_valid_slice(f)]
        picked = random.sample(files, min(SAMPLES_PER_DATASET, len(files)))
        axis_store.extend(picked)

    rng.shuffle(axis_store)
    samples_store[axis] = axis_store

    for fpath in axis_store:
        img_clean = normalize(load_slice(fpath))
        img_rand, aug = randomize_orientation(img_clean, rng)
        pol = detect_polarity(img_rand)
        atlas_slice = find_best_atlas_slice(img_rand, axis)

        t0 = time.perf_counter()
        rotated, k, angle_deg, rot_ncc, ncc_margin = best_rotation(img_rand, atlas_slice)
        if axis == "sagittal":
            final, flip = sagittal_geometric_resolve(rotated)
        else:
            final, flip = resolve_flip(rotated, atlas_slice)

        metrics = compute_metrics(final, atlas_slice, ncc_margin)
        latency_ms = (time.perf_counter() - t0) * 1000.0

        axis_samples.append({
            "input_randomized": img_rand,
            "original_clean": img_clean,
            "reoriented": final,
            "angle_deg": angle_deg,
            "flip": flip,
            "polarity": pol,
            "filename": fpath.name,
            "metrics": metrics,
            "latency_ms": latency_ms,
            "aug": aug,
        })

        m = metrics
        print(f"  [{axis}] {fpath.name[:30]:30s}  pol={pol:+d}  "
              f"rot={angle_deg:5.0f}°  flip={str(flip):5}  "
              f"dice={m['dice']:.3f}  hd={m['hausdorff_norm']:.3f}  "
              f"ncc={m['ncc']:.3f}  margin={m['ncc_margin']:.3f}  "
              f"lat={latency_ms:.1f}ms  aug={aug}")

    samples[axis] = axis_samples
    print(f"  ── {axis}: {len(axis_samples)} done ──\n")

In [ ]:
hdr = f"{'Axis':<12} {'File':<26} {'Dice':>6} {'HD_norm':>8} {'NCC':>7} {'Margin':>8} {'Latency(ms)':>12}"
print(hdr); print("-" * len(hdr))
for axis, axis_samples in samples.items():
    for s in axis_samples:
        m = s["metrics"]
        print(f"{axis:<12} {s['filename'][:26]:<26} "
              f"{m['dice']:>6.3f} {m['hausdorff_norm']:>8.3f} "
              f"{m['ncc']:>7.3f} {m['ncc_margin']:>8.3f} {s['latency_ms']:>12.1f}")

In [ ]:
axes_list = list(samples.keys())
n_cols = 6

for axis in axes_list:
    axis_samples = samples[axis]
    n = len(axis_samples)
    n_rows = int(np.ceil(n / 3))

    fig, grid = plt.subplots(n_rows * 2, n_cols, figsize=(n_cols * 3, n_rows * 2 * 3))
    if n_rows * 2 == 1:
        grid = grid[np.newaxis, :]

    for idx, s in enumerate(axis_samples):
        row_pair = idx // 3
        col_pair = (idx % 3) * 2
        m = s["metrics"]

        grid[row_pair * 2, col_pair].imshow(s["input_randomized"], cmap="gray")
        grid[row_pair * 2, col_pair].set_title(f"RAND\n{s['filename'][:15]}", fontsize=6)
        grid[row_pair * 2, col_pair].axis("off")

        grid[row_pair * 2, col_pair + 1].imshow(s["reoriented"], cmap="gray")
        grid[row_pair * 2, col_pair + 1].set_title(
            f"REORI rot={s['angle_deg']:.0f}° fl={s['flip']}\n"
            f"dice={m['dice']:.2f} ncc={m['ncc']:.2f}", fontsize=6)
        grid[row_pair * 2, col_pair + 1].axis("off")

    for idx in range(len(axis_samples), n_rows * 3):
        row_pair = idx // 3
        col_pair = (idx % 3) * 2
        grid[row_pair * 2, col_pair].axis("off")
        grid[row_pair * 2, col_pair + 1].axis("off")

    for col in range(n_cols):
        for row in range(1, n_rows * 2, 2):
            grid[row, col].axis("off")

    plt.suptitle(f"{axis} — Randomized vs Reoriented", fontsize=11)
    plt.tight_layout()
    plt.show()